# Análise inteligente de glosas médicas

Este notebook utiliza Python para analisar, priorizar e preparar indicadores de glosas médicas para interpretação com IA generativa e visualização no Power BI.

In [ ]:
import pandas as pd

arquivo = "/content/faturamento_consolidado.csv"

df = pd.read_csv(
    arquivo,
    sep=None,
    engine="python",
    decimal=",",
    encoding="utf-8-sig"
)

print(f"Quantidade de linhas: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")

df.head()

## Preparação e validação dos dados

Nesta etapa, os campos financeiros e datas são convertidos para os tipos adequados. Também é calculado o valor de glosa ainda não recuperado.

In [ ]:
# Colunas financeiras
colunas_valores = [
    "valor_apresentado",
    "valor_pago",
    "valor_glosado",
    "valor_recuperado"
]

for coluna in colunas_valores:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

# Datas
df["data_atendimento"] = pd.to_datetime(
    df["data_atendimento"],
    dayfirst=True,
    errors="coerce"
)

# Identificadores tratados como texto
df["numero_guia_prestador"] = df["numero_guia_prestador"].astype("string")
df["codigo_tuss"] = df["codigo_tuss"].astype("string")

# Categoria para itens sem glosa
df["categoria_glosa"] = df["categoria_glosa"].fillna("Sem glosa")

# Novos campos analíticos
df["valor_pendente_recuperacao"] = (
    df["valor_glosado"] - df["valor_recuperado"]
).clip(lower=0)

df["item_glosado"] = df["valor_glosado"] > 0

# Validação
print(f"Registros: {len(df)}")
print(f"Total apresentado: R$ {df['valor_apresentado'].sum():,.2f}")
print(f"Total glosado: R$ {df['valor_glosado'].sum():,.2f}")
print(f"Total recuperado: R$ {df['valor_recuperado'].sum():,.2f}")
print(
    f"Total pendente de recuperação: "
    f"R$ {df['valor_pendente_recuperacao'].sum():,.2f}"
)

## Priorização das glosas

Foi criado um score de 0 a 100 considerando o valor pendente de recuperação e o estágio do recurso. O objetivo é direcionar a análise humana para os itens com maior impacto e possibilidade de ação.

In [ ]:
import numpy as np

# Considerar somente glosas com valor ainda pendente
glosas = df[
    (df["item_glosado"]) &
    (df["valor_pendente_recuperacao"] > 0)
].copy()

# Percentil do valor pendente: quanto maior o valor, maior o score
glosas["score_valor"] = (
    glosas["valor_pendente_recuperacao"]
    .rank(pct=True, method="average") * 100
)

# Padronização do texto do status
status = (
    glosas["status_recurso"]
    .fillna("")
    .str.lower()
    .str.strip()
)

# Score de possibilidade de ação
condicoes = [
    status.str.contains("não recorrida|não apresentado", regex=True),
    status.str.contains("em análise", regex=False),
    status.str.contains("parcial", regex=False),
    status.str.contains("negado|indeferido", regex=True),
    status.str.contains("aceito|deferido integral", regex=True)
]

scores_status = [100, 70, 50, 30, 10]

glosas["score_status"] = np.select(
    condicoes,
    scores_status,
    default=40
)

# Score final
glosas["score_prioridade"] = (
    glosas["score_valor"] * 0.70 +
    glosas["score_status"] * 0.30
).round(1)

# Classificação
glosas["prioridade"] = np.select(
    [
        glosas["score_prioridade"] >= 75,
        glosas["score_prioridade"] >= 50
    ],
    [
        "Alta",
        "Média"
    ],
    default="Baixa"
)

# Ranking
ranking_glosas = glosas.sort_values(
    "score_prioridade",
    ascending=False
)

colunas_ranking = [
    "operadora",
    "numero_guia_prestador",
    "data_atendimento",
    "descricao_procedimento",
    "categoria_glosa",
    "status_recurso",
    "valor_glosado",
    "valor_recuperado",
    "valor_pendente_recuperacao",
    "score_prioridade",
    "prioridade"
]

ranking_glosas[colunas_ranking].head(10)

## Tabelas analíticas

São gerados resumos por operadora, categoria de glosa e nível de prioridade, além do ranking detalhado das glosas.

In [ ]:
# Resumo por operadora
resumo_operadoras = (
    df.groupby("operadora", as_index=False)
      .agg(
          total_itens=("numero_guia_prestador", "size"),
          total_apresentado=("valor_apresentado", "sum"),
          total_glosado=("valor_glosado", "sum"),
          total_recuperado=("valor_recuperado", "sum"),
          total_pendente=("valor_pendente_recuperacao", "sum")
      )
)

resumo_operadoras["taxa_glosa_percentual"] = (
    resumo_operadoras["total_glosado"]
    / resumo_operadoras["total_apresentado"]
    * 100
).round(2)

# Resumo por categoria
resumo_categorias = (
    glosas.groupby("categoria_glosa", as_index=False)
          .agg(
              quantidade_itens=("numero_guia_prestador", "size"),
              total_glosado=("valor_glosado", "sum"),
              total_recuperado=("valor_recuperado", "sum"),
              total_pendente=("valor_pendente_recuperacao", "sum")
          )
          .sort_values("total_glosado", ascending=False)
)

# Resumo por prioridade
resumo_prioridades = (
    ranking_glosas.groupby("prioridade", as_index=False)
                  .agg(
                      quantidade_itens=("numero_guia_prestador", "size"),
                      total_pendente=("valor_pendente_recuperacao", "sum"),
                      score_medio=("score_prioridade", "mean")
                  )
)

ordem_prioridade = {"Alta": 1, "Média": 2, "Baixa": 3}
resumo_prioridades["ordem"] = resumo_prioridades["prioridade"].map(
    ordem_prioridade
)

resumo_prioridades = (
    resumo_prioridades
    .sort_values("ordem")
    .drop(columns="ordem")
)

# Exportação
ranking_glosas[colunas_ranking].to_csv(
    "/content/ranking_prioridade_glosas.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_operadoras.to_csv(
    "/content/resumo_operadoras.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_categorias.to_csv(
    "/content/resumo_categorias_glosa.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_prioridades.to_csv(
    "/content/resumo_prioridades.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos gerados com sucesso.")
display(resumo_prioridades)

## Preparação do contexto para IA generativa

Os indicadores agregados e as glosas prioritárias são transformados em um arquivo estruturado. A IA deve utilizar somente esses dados e suas recomendações devem ser revisadas por uma pessoa.

In [ ]:
import json

# Dados estruturados para a IA
contexto_ia = {
    "projeto": "Análise inteligente de glosas médicas",
    "observacao": "Base simulada para fins educacionais e de portfólio.",
    "periodo": {
        "inicio": str(df["data_atendimento"].min().date()),
        "fim": str(df["data_atendimento"].max().date())
    },
    "indicadores": {
        "total_registros": int(len(df)),
        "total_apresentado": round(float(df["valor_apresentado"].sum()), 2),
        "total_glosado": round(float(df["valor_glosado"].sum()), 2),
        "total_recuperado": round(float(df["valor_recuperado"].sum()), 2),
        "total_pendente": round(
            float(df["valor_pendente_recuperacao"].sum()), 2
        ),
        "taxa_glosa_percentual": round(
            float(
                df["valor_glosado"].sum()
                / df["valor_apresentado"].sum()
                * 100
            ),
            2
        )
    },
    "principais_categorias": json.loads(
        resumo_categorias.head(5).round(2).to_json(
            orient="records",
            force_ascii=False
        )
    ),
    "resumo_prioridades": json.loads(
        resumo_prioridades.round(2).to_json(
            orient="records",
            force_ascii=False
        )
    ),
    "glosas_prioritarias": json.loads(
        ranking_glosas[colunas_ranking]
        .head(10)
        .to_json(
            orient="records",
            force_ascii=False,
            date_format="iso"
        )
    )
}

with open(
    "/content/contexto_ia.json",
    "w",
    encoding="utf-8"
) as arquivo_json:
    json.dump(
        contexto_ia,
        arquivo_json,
        ensure_ascii=False,
        indent=2
    )

prompt_ia = """
Você é um assistente de análise de faturamento médico.

Utilize exclusivamente os dados fornecidos no arquivo contexto_ia.json.
Não invente valores, causas ou informações clínicas.

Produza:
1. Um resumo executivo com até cinco tópicos.
2. Os três principais riscos financeiros.
3. Três recomendações operacionais priorizadas.
4. Uma observação sobre limitações dos dados.

As recomendações devem apoiar a análise humana e não devem incluir
diagnóstico, tratamento ou orientação médica.
"""

with open(
    "/content/prompt_ia.txt",
    "w",
    encoding="utf-8"
) as arquivo_prompt:
    arquivo_prompt.write(prompt_ia.strip())

print("Contexto e prompt para IA gerados com sucesso.")

In [ ]:
import os
import shutil

arquivos_saida = [
    "ranking_prioridade_glosas.csv",
    "resumo_operadoras.csv",
    "resumo_categorias_glosa.csv",
    "resumo_prioridades.csv",
    "contexto_ia.json",
    "prompt_ia.txt"
]

pasta_saida = "/content/arquivos_python"
os.makedirs(pasta_saida, exist_ok=True)

for nome in arquivos_saida:
    shutil.copy(
        f"/content/{nome}",
        f"{pasta_saida}/{nome}"
    )

shutil.make_archive(
    "/content/saidas_analise_glosas",
    "zip",
    pasta_saida
)

print("ZIP criado com sucesso.")

In [ ]:
from google.colab import files

files.download("/content/saidas_analise_glosas.zip")